# 03 --- Tool Scoping: 4-5 Tools Per Agent

**CCA Pattern**: Each agent gets 4-5 focused tools. The super agent anti-pattern (18+ tools) causes tool selection degradation.

The exam's official guidance is specific: **keep 4-5 tools per agent**. This is the exam-correct answer.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.tools.definitions import ALL_TOOL_SETS
from research_agents.anti_patterns.super_agent import SUPER_AGENT_TOOLS, get_super_agent_tool_count

## How Tool Definitions Work

Tool definitions live in `tools/definitions.py`. Each agent type has a constant list of tool dicts, for example `WEB_RESEARCHER_TOOLS`.

Each tool dict follows the Claude API format:

```python
{
    "name": "search_web",
    "description": (
        "Search the public web for information matching a query. "
        "Returns URLs, titles, and snippets. "
        "Does NOT fetch full page content (use fetch_page for that). "
        "Does NOT search internal documents or databases."
    ),
    "input_schema": { ... }
}
```

### Negative-Bound Descriptions

Every tool description includes **"Does NOT..."** clauses. This is the CCA negative-bound pattern -- it tells Claude what a tool *cannot* do, preventing misrouting. Without these, Claude might call `search_web` when it needs `parse_document`, because both involve finding information.

### The Dispatch Registry

Tool handlers are dispatched via a nested dict in `tools/handlers.py`:

```python
DISPATCH: dict[str, dict[str, Handler]] = {
    "web_researcher": {
        "search_web": handle_search_web,
        "fetch_page": handle_fetch_page,
        ...
    },
    ...
}
```

The key insight: dispatch is keyed by `(agent_type, tool_name)`. Even if two agents had a tool with the same name, they'd route to different handlers. Tools are isolated per agent at the dispatch level.

## Anti-Pattern: Super Agent with 18+ Tools

In [ ]:
# The super agent has ALL tools combined
print(f'Super agent tool count: {get_super_agent_tool_count()}')
print()
print('All tools on one agent:')
for i, tool in enumerate(SUPER_AGENT_TOOLS, 1):
    print(f'  {i:2d}. {tool["name"]:<25s} {tool["description"][:60]}...')

### Why 18+ Tools Fails

When an agent has 18 tools:
- A significant portion of attention goes to evaluating tool descriptions instead of the actual task
- Similar tools create ambiguity (e.g., `search_web` vs `cross_reference` -- both find information)
- **Better descriptions don't fix the structural problem** -- this is a key exam distractor

## The Show Truck vs. Work Truck Analogy

A useful mental model for the exam: imagine a mechanic arriving at a job site.

- **The show truck** is magnificent. It carries *every tool the mechanic owns* -- 18 drawers of sockets, 12 types of wrench, a dozen diagnostic computers, a lift and a press. Impressive. Also: every time the mechanic needs a 10mm socket, they search through 18 drawers. Every minute spent searching is a minute not spent working. The show truck looks capable but is *operationally* slow because selection cost dominates work cost.

- **The work truck** carries exactly the 4-5 tools for today's job. The mechanic reaches without looking. No selection overhead. If tomorrow's job needs different tools, they swap the loadout -- they don't bolt on more drawers.

Every specialized agent in this project is a work truck. The `super_agent` anti-pattern is a show truck. The CCA exam's correct answer is always "give the mechanic the right work truck" -- not "improve the labels on the show truck's drawers."

Concretely in this codebase: `WEB_RESEARCHER_TOOLS` is the web researcher's work truck. `DOCUMENT_ANALYZER_TOOLS` is the document analyzer's. Each set has exactly 4 tools. When the coordinator dispatches a task, it hands the *right* work truck to the *right* mechanic. That handoff is structural, not suggestive -- the dispatch registry in `tools/handlers.py` keys on `(agent_type, tool_name)`, so even if two trucks had a tool with the same name they'd route to different handlers.

## Correct Pattern: Focused Tool Sets

In [ ]:
# Each agent has 4 focused tools
print('Focused tool sets:')
for agent_type, tools in ALL_TOOL_SETS.items():
    tool_names = [t['name'] for t in tools]
    print(f'  {agent_type:<20s} ({len(tools)} tools): {", ".join(tool_names)}')

In [ ]:
from helpers import compare_results

compare_results(
    {'total_tools': get_super_agent_tool_count(), 'tools_per_agent': get_super_agent_tool_count(), 'agents': 1},
    {'total_tools': sum(len(t) for t in ALL_TOOL_SETS.values()), 'tools_per_agent': 4, 'agents': len(ALL_TOOL_SETS)},
)

## CCA Exam Tip

> The super agent question is one of the most reliable on the CCA exam.
> Answer choices will include:
> - 'improve tool descriptions' -- WRONG
> - 'add a tool-selection preprocessing step' -- WRONG
> - **'decompose into specialized subagents with 4-5 tools each' -- CORRECT**
> The improvement is architectural, not descriptive.

*Why architectural?* The root cause is **attention fragmentation** -- an 18-tool menu dilutes the model's focus regardless of how well each tool is described. Sharper descriptions are a band-aid; fewer, scoped tools fix the mechanism.